# 🏷️ Consolidate All Labels

This notebook consolidates depression labels from all datasets into a single master CSV.

**Datasets Included:**
1.  **DAIC-WOZ** (IDs 300-492)
2.  **Extended-DAIC** (IDs 600+)
3.  **EATD-Corpus** (Mandarin, extracted from folders)
4.  **LMVD** (Large-scale Multimodal, based on ID ranges)

**Output:**
- `all_labels.csv` saved to `H5_OmniFusion_Output` root.
- Compatible with `H5_OmniFusion_Dataset` loader.

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

# Define base paths
BASE_DIR = Path("/content/drive/MyDrive/DAIC-WOZ_Datasets")
OUTPUT_DIR = BASE_DIR / "H5_OmniFusion_Output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Drive mounted. Base DB: {BASE_DIR}")

In [ ]:
import pandas as pd
import numpy as np
import glob

def find_file(filename, search_path):
    """Recursively find a file in the directory."""
    results = list(search_path.rglob(filename))
    if results:
        return results[0]
    return None

def clean_pid(pid):
    """Normalize Participant ID to string."""
    return str(int(pid)) if str(pid).isdigit() else str(pid)

print("✅ Utilities defined")

In [ ]:
# Step 2: Load DAIC-WOZ & Extended-DAIC Labels
# Usually in 'detailed_lables.csv' (Note spelling: lables)

print("🔍 Searching for detailed_lables.csv...")
label_csv = find_file("detailed_lables.csv", BASE_DIR)

if not label_csv:
    print("⚠️ 'detailed_lables.csv' not found! Trying 'detailed_labels.csv'...")
    label_csv = find_file("detailed_labels.csv", BASE_DIR)

daic_records = []

if label_csv:
    print(f"📄 Found: {label_csv}")
    df = pd.read_csv(label_csv)
    
    # Check columns
    # Needs: Participant, PHQ8_Binary, PHQ8_Score (or similar)
    
    for _, row in df.iterrows():
        try:
            pid = clean_pid(row['Participant'])
            
            # Determine Source
            pid_num = int(pid) if pid.isdigit() else 0
            if 300 <= pid_num <= 499:
                source = 'DAIC-WOZ'
            elif 600 <= pid_num <= 999:
                source = 'Extended-DAIC'
            else:
                source = 'Unknown-DAIC'
                
            # Get Scores
            phq8 = row.get('PHQ8_Score', row.get('PHQ8_Total', row.get('Depression_severity', 0)))
            label = row.get('PHQ8_Binary', row.get('Depression_label', row.get('PHQ8_Diagnosis', 0)))
            
            daic_records.append({
                'Participant_ID': pid,
                'PHQ8_Score': int(phq8) if not pd.isna(phq8) else 0,
                'Depression_Label': int(label) if not pd.isna(label) else 0,
                'Source': source
            })
        except Exception as e:
            continue
            
    print(f"✅ Loaded {len(daic_records)} DAIC/Extended records")
else:
    print("❌ detailed_lables.csv NOT FOUND.")


In [ ]:
# Step 3: Load EATD-Corpus Labels
# Labels are in 'label.txt' inside each participant folder
# NOTE: 'label.txt' in EATD often contains SDS score (e.g., '53.0') instead of binary.
# SDS Cutoff: >= 53 indicates depression.

print("🔍 Scanning EATD-Corpus...")
eatd_dir = BASE_DIR / "EATD-Corpus" / "EATD-Corpus"
if not eatd_dir.exists():
    eatd_dir = BASE_DIR / "EATD-Corpus"  # Try parent

eatd_records = []

if eatd_dir.exists():
    participant_folders = [f for f in eatd_dir.glob("*") if f.is_dir()]
    
    for folder in participant_folders:
        label_file = folder / "label.txt"
        if label_file.exists():
            try:
                content = ""
                with open(label_file, 'r') as f:
                    content = f.read().strip()
                    
                # Handle SDS Score (e.g., "38.0" -> 38.0)
                score = float(content)
                
                # SDS Depression Threshold is typically 53
                is_depressed = 1 if score >= 53 else 0
                    
                pid = folder.name
                
                eatd_records.append({
                    'Participant_ID': pid,
                    'PHQ8_Score': int(score), # Using SDS score as proxy for score column
                    'Depression_Label': is_depressed,
                    'Source': 'EATD-Corpus'
                })
            except Exception as e:
                print(f"Error reading {label_file}: {e} (Content: '{content}')")
                
    print(f"✅ Loaded {len(eatd_records)} EATD records")
else:
    print(f"❌ EATD Directory not found: {eatd_dir}")


In [ ]:
# Step 4: Load LMVD Labels using Hardcoded Ranges from README
# Depression Ranges: 001-601, 1117-1423
# Normal Ranges: 0602-1116, 1425-1824

print("🔍 Scanning LMVD_Raw for participant IDs...")
lmvd_dir = BASE_DIR / "LMVD_Raw"
if not lmvd_dir.exists():
    lmvd_dir = BASE_DIR / "LMVD"  # Try alt name

lmvd_records = []

if lmvd_dir.exists():
    # Get unique ID stems from filenames (e.g. 001.csv -> 1)
    unique_ids = set()
    for f in lmvd_dir.glob("*.*"): # Scan all files
        if f.stem.isdigit():
            unique_ids.add(int(f.stem))
            
    print(f"Found {len(unique_ids)} LMVD participants")

    for pid_num in sorted(list(unique_ids)):
        is_depressed = 0
        # Logic from README
        if (1 <= pid_num <= 601) or (1117 <= pid_num <= 1423):
            is_depressed = 1
        
        pid_str = str(pid_num).zfill(3)
        
        lmvd_records.append({
            'Participant_ID': pid_str,
            'PHQ8_Score': 15 if is_depressed else 3,
            'Depression_Label': is_depressed,
            'Source': 'LMVD'
        })
    print(f"✅ Generated labels for {len(lmvd_records)} LMVD records")
else:
    print(f"❌ LMVD Directory not found: {lmvd_dir}")

In [ ]:
# Step 5: Merge and Save

all_records = daic_records + eatd_records + lmvd_records

if all_records:
    final_df = pd.DataFrame(all_records)
    
    # Ensure columns
    cols = ['Participant_ID', 'PHQ8_Score', 'Depression_Label', 'Source']
    final_df = final_df[cols]
    
    output_path = OUTPUT_DIR / "all_labels.csv"
    final_df.to_csv(output_path, index=False)
    
    print("="*50)
    print("🏆 CONSOLIDATION COMPLETE")
    print("="*50)
    print(f"💾 Saved to: {output_path}")
    print(f"📦 Total Samples: {len(final_df)}")
    print("\n📊 Breakdown by Source:")
    print(final_df['Source'].value_counts())
    print("\n📊 Breakdown by Label:")
    print(final_df['Depression_Label'].value_counts())
    
    # Detailed View
    print("\n📊 Detailed Breakdown (Source vs Label):")
    try:
        print(pd.crosstab(final_df['Source'], final_df['Depression_Label']))
    except Exception as e:
        print(f"Could not print detailed breakdown: {e}")
    
    # Validity check
    depressed_count = final_df[final_df['Depression_Label'] == 1].shape[0]
    if depressed_count == 0:
        print("\n⚠️ WARNING: No depressed samples found! Check source files.")
    else:
        print(f"\n✅ Found {depressed_count} depressed samples.")
        
else:
    print("❌ No records found to merge!")